<a href="https://colab.research.google.com/github/rhodes-byu/stat-486/blob/main/notebooks/13-dimension-reduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a><p><b></b></p>


# Dimensionality Reduction Tutorial: Music Emotion Features

This notebook turns the lecture on dimension reduction into a hands-on investigation.
We will use acoustic features from songs to study how lower-dimensional representations can help us **visualize**, **summarize**, and sometimes **model** high-dimensional data.

We will follow the same big ideas from the slides:
- high-dimensional data creates distance and sparsity problems
- **PCA** is a linear projection method built around explained variance
- **t-SNE** and **UMAP** are nonlinear embedding methods aimed mostly at visualization
- extracted features can help or hurt a downstream supervised model depending on how many dimensions we keep

## Learning goals
By the end of this notebook, you should be able to:
1. Explain one consequence of the curse of dimensionality.
2. Interpret a scree plot and cumulative explained variance plot from PCA.
3. Color a low-dimensional embedding by labels or original variables and describe what you see.
4. Compare model performance across different numbers of extracted PCA features.


## Quick Review from the Slides

The PDF emphasizes four main ideas:
- In high dimensions, data points become sparse and distances become less informative.
- PCA finds orthogonal directions that preserve as much variance as possible.
- The fraction of variance explained by each principal component helps us decide how many components to keep.
- t-SNE and UMAP are useful for visualization because they focus on preserving local neighborhoods, but their axes are not directly interpretable like PCA loadings.

This notebook is organized to let you test each of those ideas yourself.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import pairwise_distances
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.float_format = "{:,.3f}".format

RANDOM_STATE = 42
LABEL_COLORS = {"happy": "#1f77b4", "sad": "#d62728"}


In [ ]:
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

data_path = ROOT / "data" / "music_emotion.csv"
df = pd.read_csv(data_path)
df.head()


## 1. Meet the Dataset

`music_emotion.csv` contains engineered audio features for **4,138 songs** labeled `happy` or `sad`.
The predictors include spectral summaries and MFCC-based features, which gives us a moderate-sized numeric feature space that is large enough for PCA and embedding methods to be meaningful.

### Student prompts
- Which columns look like predictors, and which columns should not be used as model features?
- Which features seem likely to have very different scales?
- Why would scaling matter before PCA or t-SNE?


In [ ]:
feature_cols = [col for col in df.columns if col not in ["filename", "label"]]
X = df[feature_cols]
y = df["label"]

print(f"Rows: {df.shape[0]:,}")
print(f"Numeric features: {len(feature_cols)}")
print(f"Missing values: {int(df.isna().sum().sum())}")
print()
print("Class balance:")
print(y.value_counts())

X.describe().T[["mean", "std", "min", "max"]].head(10)


## 2. A Small Curse-of-Dimensionality Experiment

The slides note that nearest and farthest points become harder to distinguish in high dimensions.
We can explore that idea by sampling a subset of songs, measuring pairwise distances, and comparing the average nearest and farthest distances as we increase the number of features.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

rng = np.random.default_rng(RANDOM_STATE)
row_idx = rng.choice(X_scaled.shape[0], size=250, replace=False)
feature_order = rng.permutation(X_scaled.shape[1])

records = []
for n_features in [2, 5, 10, 20, X_scaled.shape[1]]:
    X_sub = X_scaled[row_idx][:, feature_order[:n_features]]
    D = pairwise_distances(X_sub)
    np.fill_diagonal(D, np.nan)

    nearest = np.nanmin(D, axis=1)
    farthest = np.nanmax(D, axis=1)

    records.append(
        {
            "n_features": n_features,
            "mean_nearest_distance": nearest.mean(),
            "mean_farthest_distance": farthest.mean(),
            "nearest_to_farthest_ratio": nearest.mean() / farthest.mean(),
            "relative_contrast": (farthest.mean() - nearest.mean()) / nearest.mean(),
        }
    )

distance_summary = pd.DataFrame(records)
distance_summary


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(
    distance_summary["n_features"],
    distance_summary["mean_nearest_distance"],
    marker="o",
    linewidth=2,
    label="Average nearest distance",
)
ax[0].plot(
    distance_summary["n_features"],
    distance_summary["mean_farthest_distance"],
    marker="o",
    linewidth=2,
    label="Average farthest distance",
)
ax[0].set_xlabel("Number of features used")
ax[0].set_ylabel("Distance")
ax[0].set_title("Nearest vs. farthest distances")
ax[0].legend()

ax[1].plot(
    distance_summary["n_features"],
    distance_summary["nearest_to_farthest_ratio"],
    marker="o",
    linewidth=2,
    color="darkred",
)
ax[1].set_xlabel("Number of features used")
ax[1].set_ylabel("Nearest / farthest distance")
ax[1].set_title("Distance contrast shrinks in higher dimensions")

plt.tight_layout()


### Student prompts
- What happens to the gap between the nearest and farthest points as the number of features increases?
- Why is that a problem for algorithms such as KNN or clustering methods that depend on distances?
- If you rerun this cell with a different random seed or different subset of features, does the same broad pattern remain?


## 3. PCA: Explained Variance and Feature Extraction

PCA is a **projection** method. After centering and scaling the features, it finds orthogonal directions that capture the largest possible variance.
This section mirrors the PCA part of the lecture: fit PCA, inspect explained variance, and decide how many components to keep.


In [ ]:
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)

explained_variance = pd.DataFrame(
    {
        "component": np.arange(1, len(feature_cols) + 1),
        "explained_variance_ratio": pca_full.explained_variance_ratio_,
        "cumulative_explained_variance": np.cumsum(pca_full.explained_variance_ratio_),
    }
)

threshold_summary = pd.DataFrame(
    {
        "variance_threshold": [0.70, 0.80, 0.90, 0.95],
        "components_needed": [
            np.searchsorted(explained_variance["cumulative_explained_variance"], threshold) + 1
            for threshold in [0.70, 0.80, 0.90, 0.95]
        ],
    }
)

display(threshold_summary)
explained_variance.head(10)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].bar(
    explained_variance["component"],
    explained_variance["explained_variance_ratio"],
    color="#4c72b0",
)
ax[0].set_xlabel("Principal component")
ax[0].set_ylabel("Explained variance ratio")
ax[0].set_title("Scree plot")

ax[1].plot(
    explained_variance["component"],
    explained_variance["cumulative_explained_variance"],
    marker="o",
    linewidth=2,
    color="#dd8452",
)
for threshold in [0.70, 0.80, 0.90, 0.95]:
    ax[1].axhline(threshold, linestyle="--", color="gray", alpha=0.5)
ax[1].set_xlabel("Number of components")
ax[1].set_ylabel("Cumulative explained variance")
ax[1].set_title("How many PCs should we keep?")
ax[1].set_ylim(0, 1.02)

plt.tight_layout()


### Student prompts
- About how many principal components are needed to explain 80% of the variance?
- Is there an obvious "elbow" in the scree plot?
- If you had to pick a compact feature set for a model, what range of components would you consider first, and why?


In [ ]:
loadings = pd.DataFrame(
    pca_full.components_.T,
    index=feature_cols,
    columns=[f"PC{i}" for i in range(1, len(feature_cols) + 1)],
)

def top_loadings(component, top_n=8):
    out = pd.DataFrame(
        {
            "loading": loadings[component],
            "abs_loading": loadings[component].abs(),
        }
    )
    return out.sort_values("abs_loading", ascending=False).head(top_n)

print("Top loadings for PC1")
display(top_loadings("PC1"))
print("Top loadings for PC2")
display(top_loadings("PC2"))


The loadings tell us which original variables contribute most strongly to each principal component.
PCA axes are linear combinations of the original features, so unlike t-SNE, the directions themselves are interpretable.

### Student prompts
- Which original features seem most important for PC1 and PC2?
- Do the signs of the loadings matter? What does a positive or negative loading mean here?
- Which pairs of variables seem like they might be carrying similar information?


In [ ]:
pca_plot_df = pd.DataFrame(X_pca_full[:, :2], columns=["PC1", "PC2"])
pca_plot_df["label"] = y.values
for col in ["spectral_centroid", "chroma_stft", "zero_crossing_rate", "rolloff", "mfcc2"]:
    pca_plot_df[col] = X[col].values

pca_plot_df.head()


In [ ]:
feature_to_color = "spectral_centroid"  # Try: "rolloff", "zero_crossing_rate", "mfcc2", "chroma_stft"


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

for label_name, color in LABEL_COLORS.items():
    mask = pca_plot_df["label"] == label_name
    ax[0].scatter(
        pca_plot_df.loc[mask, "PC1"],
        pca_plot_df.loc[mask, "PC2"],
        s=18,
        alpha=0.7,
        color=color,
        label=label_name,
    )

ax[0].set_title("First two principal components")
ax[0].set_xlabel("PC1")
ax[0].set_ylabel("PC2")
ax[0].legend(title="label")

sc = ax[1].scatter(
    pca_plot_df["PC1"],
    pca_plot_df["PC2"],
    c=pca_plot_df[feature_to_color],
    cmap="viridis",
    s=18,
    alpha=0.7,
)
ax[1].set_title(f"PCA map colored by {feature_to_color}")
ax[1].set_xlabel("PC1")
ax[1].set_ylabel("PC2")
plt.colorbar(sc, ax=ax[1], label=feature_to_color)

plt.tight_layout()


### Student prompts
- How much label separation do you see in the first two PCs?
- Change `feature_to_color` and look for smooth gradients. Which original variables seem most aligned with PC1 or PC2?
- Does a variable with strong PCA loadings also produce a clear color gradient in the PCA scatterplot?


## 4. PCA Features in a Supervised Model

The slides mention that principal components can be used as features in another machine learning model.
Here we will compare logistic regression accuracy as we vary the number of retained principal components.

One important idea to watch for: **the number of components that explains a lot of variance is not automatically the number that gives the best classification accuracy.**


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
component_grid = [2, 4, 6, 8, 10, 12, 16, 20, len(feature_cols)]

baseline_pipe = Pipeline(
    [
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000)),
    ]
)
baseline_score = cross_val_score(
    baseline_pipe,
    X,
    y,
    cv=cv,
    scoring="accuracy",
    n_jobs=1,
).mean()

model_records = []
for n_components in component_grid:
    pipe = Pipeline(
        [
            ("scale", StandardScaler()),
            ("pca", PCA(n_components=n_components)),
            ("model", LogisticRegression(max_iter=3000)),
        ]
    )
    scores = cross_val_score(
        pipe,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        n_jobs=1,
    )
    model_records.append(
        {
            "n_components": n_components,
            "mean_cv_accuracy": scores.mean(),
            "std_cv_accuracy": scores.std(),
        }
    )

model_results = pd.DataFrame(model_records)
print(f"Baseline logistic regression using all original standardized features: {baseline_score:.3f}")
model_results


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(
    model_results["n_components"],
    model_results["mean_cv_accuracy"],
    yerr=model_results["std_cv_accuracy"],
    marker="o",
    linewidth=2,
    capsize=4,
    label="PCA features",
)
ax.axhline(
    baseline_score,
    linestyle="--",
    color="gray",
    label="All original features",
)
ax.set_xlabel("Number of principal components")
ax.set_ylabel("Cross-validated accuracy")
ax.set_title("How many PCA features help this classifier?")
ax.legend()
plt.tight_layout()


### Student prompts
- At what point does accuracy stop improving very much?
- How does the best PCA-based model compare with the model that uses all original features?
- Does the component count that gives about 80% or 90% explained variance also seem near the best predictive performance?
- Why might a variance-maximizing method fail to perfectly match a prediction-maximizing objective?


## 5. PCA vs. t-SNE for Visualization

PCA gives linear combinations of the original features, so its axes are interpretable and it can transform new data easily.
t-SNE is different: it is designed mainly to preserve **local neighborhoods** in a low-dimensional map.

Because t-SNE is slower, we will run it on a balanced sample of the songs.


In [ ]:
sample_parts = []
for _, group in df.groupby("label"):
    sample_parts.append(group.sample(n=min(len(group), 500), random_state=RANDOM_STATE))

sample_df = pd.concat(sample_parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

X_sample = sample_df[feature_cols]
y_sample = sample_df["label"]
X_sample_scaled = StandardScaler().fit_transform(X_sample)

pca_sample = PCA(n_components=2).fit_transform(X_sample_scaled)
tsne_sample = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=RANDOM_STATE,
).fit_transform(X_sample_scaled)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

for label_name, color in LABEL_COLORS.items():
    mask = y_sample == label_name
    ax[0].scatter(
        pca_sample[mask, 0],
        pca_sample[mask, 1],
        s=20,
        alpha=0.7,
        color=color,
        label=label_name,
    )
    ax[1].scatter(
        tsne_sample[mask, 0],
        tsne_sample[mask, 1],
        s=20,
        alpha=0.7,
        color=color,
        label=label_name,
    )

ax[0].set_title("PCA on a 1,000-song sample")
ax[0].set_xlabel("PC1")
ax[0].set_ylabel("PC2")
ax[0].legend(title="label")

ax[1].set_title("t-SNE on the same sample")
ax[1].set_xlabel("t-SNE 1")
ax[1].set_ylabel("t-SNE 2")
ax[1].legend(title="label")

plt.tight_layout()


### Student prompts
- Which method makes local clusters or pockets of similar songs easier to see?
- Which method has interpretable directions that you can connect back to original variables?
- Why should we be careful about over-interpreting the distances between clusters in a t-SNE map?
- Try changing `perplexity` to 5, 30, and 50. What changes?


## 6. Optional UMAP Extension

UMAP is another nonlinear embedding method discussed in the slides.
It is often faster than t-SNE and can preserve more global structure, but it requires the optional `umap-learn` package.


In [ ]:
try:
    import umap
    has_umap = True
except ImportError:
    has_umap = False
    print("UMAP is optional here. Install `umap-learn` if you want to run this section.")

if has_umap:
    umap_embedding = umap.UMAP(
        n_components=2,
        n_neighbors=20,
        min_dist=0.1,
        random_state=RANDOM_STATE,
    ).fit_transform(X_sample_scaled)

    fig, ax = plt.subplots(figsize=(6, 5))
    for label_name, color in LABEL_COLORS.items():
        mask = y_sample == label_name
        ax.scatter(
            umap_embedding[mask, 0],
            umap_embedding[mask, 1],
            s=20,
            alpha=0.7,
            color=color,
            label=label_name,
        )

    ax.set_title("UMAP on the same 1,000-song sample")
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.legend(title="label")
    plt.tight_layout()


## Stretch Ideas

If you want to extend this notebook later, here are some good follow-up activities:
- Repeat the PCA modeling section with **KNN** or **SVM** and compare whether the preferred number of components changes.
- Run PCA **without scaling** and compare the loadings and scree plot.
- Use a different color variable in the PCA scatterplot and identify which features create the clearest gradients.
- Apply PCA to an image dataset such as handwritten digits and study **reconstruction** or **compression**.
- Compare t-SNE and UMAP over multiple hyperparameter settings and discuss which visual patterns are stable.
- Ask students to defend a choice of `k` using both explained variance and predictive performance instead of only one metric.


## Suggested Write-Up Questions

1. In your own words, how did the distance experiment illustrate the curse of dimensionality?
2. How many principal components were needed to explain 80% of the variance? Did that same number appear close to best for classification accuracy?
3. Which original feature seemed most strongly reflected in the first two principal components when you colored the PCA scatterplot?
4. What did t-SNE reveal more clearly than PCA, and what did PCA preserve better than t-SNE?
5. If you had to build a simple emotion classifier from this dataset, would you use the original features or PCA features? Defend your choice.
